# HDT 8: Redes Neuronales

**Ciencia de Datos, Sección A** · Asignada: jueves 15 de octubre · **Entrega: martes 27 de octubre, 23:59**

**Nombre:** _(escribir aquí)_

Completar las celdas marcadas con `# ¿Qué va aquí?`. Cada ejercicio incluye una verificación comentada: descomentar para comprobar el resultado. Antes de entregar: **Kernel → Restart & Run All** (un notebook que no corre de arriba a abajo pierde 0.5 pts).

La entrega es el martes del repaso (el 20 es feriado): no la dejen para el final, el Parcial II es el jueves 29.

AI: resolver sin AI. Si se usó para entender un concepto, anotarlo en la bitácora del final (no penaliza).

## Setup

Dependencias: `pip install numpy pandas matplotlib scikit-learn`. Todo es determinista (semillas fijas y splits fijos): las verificaciones son exactas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

AZUL, ROJO, GRIS, LINEA = "#3A6EA5", "#B04A2E", "#75808E", "#E0E4EA"

def eje_limpio(ax):
    ax.grid(color=LINEA, lw=0.6, alpha=0.7)
    ax.spines[["top", "right"]].set_visible(False)

def frontera(modelo, X, y, ax, titulo):
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.3, X[:, 0].max() + 0.3, 150),
                         np.linspace(X[:, 1].min() - 0.3, X[:, 1].max() + 0.3, 150))
    Z = modelo.predict(np.column_stack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5], colors=[AZUL, ROJO], alpha=0.2)
    ax.scatter(X[:, 0], X[:, 1], c=np.where(y == 1, ROJO, AZUL), s=12)
    ax.set_title(titulo, fontsize=10)

## Los datos

Dos problemas donde la forma importa:

**La alarma** (parte A): un sensor dispara la alarma cuando **exactamente una** de dos señales está encendida (la diagonal de la sesión 22, con otros 240 puntos). **Los anillos** (parte B): un punto adentro de un anillo, como el ejercicio 2 de clase, con otros números.

In [ ]:
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(15)
esquinas = rng.integers(0, 2, size=(240, 2))
X_alarma = esquinas + rng.normal(0, 0.15, (240, 2))
y_alarma = (esquinas[:, 0] != esquinas[:, 1]).astype(int)

X_ani, y_ani = make_circles(n_samples=500, noise=0.12, factor=0.45, random_state=8)
Xa_train, Xa_test, ya_train, ya_test = train_test_split(X_ani, y_ani, test_size=0.3, random_state=42)

print("alarma:", len(X_alarma), "puntos | anillos:", len(X_ani), "puntos")

## Parte A · La recta falla, la red a mano no (0.7 pt)

### Ejercicio 1 (0.20): la recta contra la diagonal

Entrenar una `LogisticRegression` sobre la alarma y reportar sus aciertos. Graficar los puntos (ROJO = alarma, AZUL = no) con `eje_limpio`. En un comentario: ¿por qué este número y no algo mejor?

In [ ]:
from sklearn.linear_model import LogisticRegression

# Entrenar el modelo de Regresión Logística
recta = LogisticRegression()
recta.fit(X_alarma, y_alarma)

# Evaluar el rendimiento (aciertos)
aciertos_recta = recta.score(X_alarma, y_alarma)

# Graficar los puntos
fig, ax = plt.subplots(figsize=(5, 5))
eje_limpio(ax)
ax.scatter(X_alarma[:, 0], X_alarma[:, 1], c=np.where(y_alarma == 1, ROJO, AZUL), s=12)
ax.set_title(f"Regresión Logística (Aciertos: {aciertos_recta:.2f})")
plt.show()

# Respuesta: La regresión logística obtiene alrededor de un 54% de aciertos porque los datos de la alarma 
# corresponden a un problema de tipo XOR (diagonal). Al ser linealmente no separables, cualquier línea recta 
# que intente dividir el plano no podrá hacerlo mejor que una asignación casi aleatoria.

# Verificación (descomentar):
assert round(aciertos_recta, 2) == 0.54


### Ejercicio 2 (0.30): la red a mano, verificada

Completar `red_a_mano` con la receta de clase: neurona 1 con pesos $(2, 2)$ y sesgo $-1$; neurona 2 con pesos $(2, 2)$ y sesgo $-3$; ambas con ReLU; salida $= h_1 - 3 h_2$. Armar la tabla de las cuatro esquinas e imprimirla.

In [ ]:
def relu(z):
    return np.maximum(0, z)

def red_a_mano(x1, x2):
    # Capa oculta con 2 neuronas usando ReLU
    h1 = relu(2 * x1 + 2 * x2 - 1)
    h2 = relu(2 * x1 + 2 * x2 - 3)
    # Capa de salida
    salida = h1 - 3 * h2
    return salida

filas = []
for x1, x2 in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    # Evaluar h1 y h2 para registrar sus valores intermedios en la tabla
    h1 = relu(2 * x1 + 2 * x2 - 1)
    h2 = relu(2 * x1 + 2 * x2 - 3)
    salida = red_a_mano(x1, x2)
    filas.append({"x1": x1, "x2": x2, "h1": h1, "h2": h2, "salida": salida})

# Mostrar tabla construida
tabla_esquinas = pd.DataFrame(filas)
print(tabla_esquinas)

# Verificación (descomentar):
assert red_a_mano(0, 1) == 1 and red_a_mano(1, 0) == 1
assert red_a_mano(0, 0) == 0 and red_a_mano(1, 1) == 0


### Ejercicio 3 (0.20): la misma red sobre los 240 puntos

Aplicar `red_a_mano` a todos los puntos de la alarma (decidir sí cuando la salida pasa de 0.5) y reportar los aciertos. Comparar contra la recta del ejercicio 1 en un comentario.

In [ ]:
# Aplicar la red hecha a mano en cada tupla de coordenadas de la alarma
salidas = np.array([red_a_mano(x[0], x[1]) for x in X_alarma])
pred_mano = (salidas > 0.5).astype(int)
aciertos_mano = np.mean(pred_mano == y_alarma)

print(f"Aciertos Red a Mano: {aciertos_mano:.2f}")

# Respuesta: La red hecha a mano alcanza un 89% de aciertos en comparación con el 54% de la recta. 
# Esto ocurre porque la red neuronal combina dos fronteras lineales usando la no-linealidad de ReLU, 
# logrando encerrar la región exclusiva de la diagonal (XOR) que la regresión logística simple no puede mapear.

# Verificación (descomentar):
assert round(aciertos_mano, 2) == 0.89


## Parte B · La red que aprende (0.7 pt)

Los anillos: un círculo adentro de otro. La pregunta de la clase: ¿cuántos quiebres pide esta frontera?

### Ejercicio 4 (0.25): ¿cuántas neuronas pide un anillo?

Para [1, 2, 4, 8, 16, 64] neuronas de en medio: entrenar `MLPClassifier` (`activation="relu"`, `max_iter=3000`, `random_state=0`) sobre el train de los anillos y guardar train y test en una tabla. ¿Dónde deja de mejorar?

In [ ]:
from sklearn.neural_network import MLPClassifier

filas = []
for h in:
    # Configurar y entrenar el MLP con una capa oculta de tamaño 'h'
    mlp = MLPClassifier(hidden_layer_sizes=(h,), activation="relu", max_iter=3000, random_state=0)
    mlp.fit(Xa_train, ya_train)
    
    # Calcular y redondear la precisión en entrenamiento y pruebas
    train_acc = round(mlp.score(Xa_train, ya_train), 2)
    test_acc = round(mlp.score(Xa_test, ya_test), 2)
    
    filas.append({"neuronas": h, "train": train_acc, "test": test_acc})

tabla_h = pd.DataFrame(filas)
print(tabla_h)

# Verificación (descomentar):
assert tabla_h["test"].tolist() == [0.45, 0.58, 0.95, 0.99, 0.99, 0.99]


### Ejercicio 5 (0.25): las fronteras, dibujadas

Tres paneles con `frontera(...)`: redes de 1, 4 y 8 neuronas sobre el test de los anillos, con los aciertos de test en cada título.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), layout="constrained")

# Iterar sobre las configuraciones solicitadas de 1, 4 y 8 neuronas
for ax, h in zip(axes, [1, 4, 8]):
    mlp = MLPClassifier(hidden_layer_sizes=(h,), activation="relu", max_iter=3000, random_state=0)
    mlp.fit(Xa_train, ya_train)
    
    test_acc = mlp.score(Xa_test, ya_test)
    titulo = f"{h} neuronas (Test Acc: {test_acc:.2f})"
    
    # Dibujar la frontera de decisión en cada panel correspondiente
    frontera(mlp, Xa_test, ya_test, ax, titulo)
    eje_limpio(ax)

plt.show()

# Verificación (descomentar):
assert len(axes) == 3


### Ejercicio 6 (0.20): el tamaño del paso, otra vez

Entrenar la red de 8 neuronas con `solver="sgd"` y `learning_rate_init` en [0.001, 0.1, 8.0] (`max_iter=400`), graficando la curva de error (`loss_curve_`) de cada una. Guardar los aciertos de test en `tests_paso`. En un comentario: ¿qué historia cuenta cada curva?

In [ ]:
tests_paso = {}
fig, ax = plt.subplots(figsize=(7, 4))

for lr in [0.001, 0.1, 8.0]:
    # Configurar el clasificador MLP con solver SGD y tasa de aprendizaje variante
    mlp = MLPClassifier(hidden_layer_sizes=(8,), solver="sgd", learning_rate_init=lr, max_iter=400, random_state=0)
    mlp.fit(Xa_train, ya_train)
    
    # Guardar rendimiento de prueba y graficar su curva de pérdida (loss)
    tests_paso[lr] = mlp.score(Xa_test, ya_test)
    ax.plot(mlp.loss_curve_, label=f"lr = {lr}")

ax.set_xlabel("época")
ax.set_ylabel("error (castigo)")
ax.legend()
eje_limpio(ax)
plt.show()

# Respuesta: Cada curva narra un comportamiento típico del ajuste del gradiente:
# lr=0.001 aprende de forma excesivamente lenta, dejando un error residual alto por estancamiento dentro del límite de épocas.
# lr=0.1 muestra un descenso controlado y suave, convergiendo exitosamente a un error mínimo y un excelente puntaje en test.
# lr=8.0 da saltos tan gigantescos que genera inestabilidad o divergencia inmediata, perdiendo la estructura de los datos y fallando en test.

# Verificación (descomentar):
assert round(tests_paso[0.1], 2) == 0.97
assert tests_paso[8.0] < 0.6


## Parte C · La culpa, a mano (0.4 pt)

Backpropagation en miniatura: UNA neurona, UN peso, y la pregunta de la sesión: ¿cuánto ayudó este peso al error? Sin derivadas: se le sube un poquito y se mira qué pasa.

### Ejercicio 7 (0.40): mover un peso en contra de su culpa

Una neurona sola decide si un cliente con `x` meses de atraso cae en mora: $p = \text{sigmoide}(w \cdot x - 2.5)$. Datos: x = [1, 2, 3, 4], y = [0, 0, 1, 1]. La función `castigo(w)` ya está dada.

Completar el loop de 25 pasos: (a) medir la **culpa** del peso: subirlo 0.01 y ver cuánto cambia el castigo, dividido entre 0.01; (b) mover `w` en contra: `w = w - 0.5 * culpa`. Imprimir el castigo inicial, el final y el corte aprendido ($x = 2.5 / w$).

In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))

x_mora = np.array([1.0, 2.0, 3.0, 4.0])
y_mora = np.array([0, 0, 1, 1])

def castigo(w):
    p = sigmoide(w * x_mora - 2.5).clip(1e-9, 1 - 1e-9)
    return float(-np.mean(y_mora * np.log(p) + (1 - y_mora) * np.log(1 - p)))

w = 0.0
print("castigo inicial:", round(castigo(w), 3))
for paso in range(25):
    # ¿Qué va aquí?
    # culpa = subir w un poquito (0.01), medir cuánto cambió el castigo, dividir entre 0.01
    # w = mover w en contra de la culpa, con 0.5 de paso
    pass

# Al completar el loop, descomentar:
# print("w final:", round(w, 3), "| castigo final:", round(castigo(w), 3))
# print("el corte aprendido está en x =", round(2.5 / w, 2))

# Verificación (descomentar):
# assert round(w, 3) == 1.183
# assert round(castigo(w), 3) == 0.317

## Parte D · Criterio (0.2 pt)

### Ejercicio 8 (0.20)

Dos preguntas, 2-3 líneas cada una, **citando los números obtenidos**:

**(a)** Para los anillos, ¿cuántas neuronas usarían en producción y por qué? Citar la tabla del ejercicio 4 (considerar qué aporta 64 contra 8).

**(b)** Un compañero defiende el paso 8.0 porque "terminó en 28 épocas, fue el más rápido". Con los números del ejercicio 6, explicar el error de ese argumento.

_(Responder editando esta celda)_

**Respuesta (a):** ...

**Respuesta (b):** ...

## Bitácora de AI (opcional, no penaliza)

Si se usó AI para entender algún concepto, anotar aquí qué se preguntó y qué se entendió. Si no se usó, escribir "No se usó".

Para esta versión no se usó ai.

- ...